# Step 03 — PCA & Clustering

This notebook explores the PCA dimensionality reduction and k-means clustering
results produced by `pipelines/03_cluster.py`.

Two cluster assignments are always produced:
- `cluster`: best k chosen by silhouette score (capped at `k_cap`, default 5)
- `balanced_cluster`: size-constrained k-means at a fixed `balanced_k` (default 5)

Downstream regime labelling uses `balanced_cluster` by default because
equal-size clusters give better per-regime statistics with limited data.

**Run `python pipelines/03_cluster.py` before executing this notebook.**

## Setup & Imports

In [1]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
import pandas as pd
import matplotlib.pyplot as plt
from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR, plotting
setup_logging("INFO")
log = logging.getLogger("03_clustering")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=False)

In [2]:
import subprocess
from pathlib import Path

def run_step_if_needed(step: int, required_paths: list, auto_run: bool = True) -> bool:
    """Run the pipeline step if any required output files are missing."""
    missing = [p for p in required_paths if not Path(p).exists()]
    if not missing:
        return True
    print(f"Missing: {[str(p) for p in missing]}")
    scripts = sorted(Path("../pipelines").glob(f"{step:02d}_*.py"))
    if not scripts:
        print(f"No pipeline script found for step {step}.")
        return False
    script = scripts[0]
    if not auto_run:
        print(f"  → Run: python {script}")
        return False
    print(f"  → Running {script.name} ...")
    result = subprocess.run(["python", str(script)], capture_output=True, text=True, cwd="..")
    out = result.stdout
    if len(out) > 4000:
        out = out[:2000] + "\n...\n" + out[-2000:]
    print(out)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        return False
    print(f"  ✓ Step {step} complete.")
    return True

# Run clustering step if regime files are missing
run_step_if_needed(3, [
    DATA_DIR / "regimes" / "cluster_labels.parquet",
    DATA_DIR / "regimes" / "pca_components.parquet",
])

True

## Load Clustering Artifacts

Three parquet files are loaded from `data/regimes/`:
- `cluster_labels.parquet` — quarter → cluster assignment (both `cluster` and `balanced_cluster`)
- `pca_components.parquet` — the 5 PCA components for each quarter
- `kmeans_scores.parquet` — silhouette / Calinski-Harabasz / Davies-Bouldin scores for each k

In [3]:
REGIMES_DIR = DATA_DIR / "regimes"

labels = None
pca_df = None
scores = None

try:
    labels = pd.read_parquet(REGIMES_DIR / "cluster_labels.parquet")
    print(f"Labels loaded:         {labels.shape[0]} quarters, columns: {list(labels.columns)}")
except FileNotFoundError:
    print("ERROR: cluster_labels.parquet not found. Run pipelines/03_cluster.py first.")

try:
    pca_df = pd.read_parquet(REGIMES_DIR / "pca_components.parquet")
    print(f"PCA components loaded: {pca_df.shape[0]} quarters x {pca_df.shape[1]} components")
except FileNotFoundError:
    print("ERROR: pca_components.parquet not found. Run pipelines/03_cluster.py first.")

try:
    scores = pd.read_parquet(REGIMES_DIR / "kmeans_scores.parquet")
    print(f"K-sweep scores loaded: {scores.shape[0]} rows, columns: {list(scores.columns)}")
except FileNotFoundError:
    print("ERROR: kmeans_scores.parquet not found. Run pipelines/03_cluster.py first.")

Labels loaded:         304 quarters, columns: ['cluster', 'balanced_cluster', 'market_code']
PCA components loaded: 304 quarters x 5 components
K-sweep scores loaded: 11 rows, columns: ['k', 'inertia', 'silhouette', 'calinski', 'davies_bouldin']


## PCA Explained Variance

The pipeline uses a fixed 5 PCA components (per `n_pca_components` in settings.yaml).
Explained variance is reported if stored separately.

In [4]:
if pca_df is not None:
    print(f"PCA columns: {list(pca_df.columns)}")
    print(f"Shape:       {pca_df.shape}")
    print()
    # Check for separately-stored explained variance
    ev_path = REGIMES_DIR / "pca_explained_variance.parquet"
    try:
        ev = pd.read_parquet(ev_path)
        print("PCA explained variance:")
        display(ev)
    except FileNotFoundError:
        print("(pca_explained_variance.parquet not found — variance not stored separately)")
    print()
    print("First 5 rows of PCA components:")
    display(pca_df.head())

PCA columns: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
Shape:       (304, 5)

(pca_explained_variance.parquet not found — variance not stored separately)

First 5 rows of PCA components:


,PC1,PC2,PC3,PC4,PC5
date,,,,,
1950-03-31,2.711780,-1.167017,3.607649,2.798279,1.089949
1950-06-30,2.440854,-1.800320,4.244305,1.684553,0.476226
1950-09-30,1.788419,-2.259959,4.483964,0.224586,-0.257439
1950-12-31,0.576020,-2.470894,4.284812,-2.138648,-1.730186
1951-03-31,-0.650086,-2.033331,3.460589,-4.033916,-2.985986


## K-Sweep Elbow Curves

Three metrics are plotted across the range of k values tested (default k=2..12).
The vertical dashed line marks the automatically chosen best k (max silhouette).

In [5]:
if scores is not None:
    print("K-sweep scores:")
    display(scores)

    if "silhouette" in scores.columns and "k" in scores.columns:
        best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
        print(f"\nBest k (max silhouette): {best_k}")
        plotting.plot_elbow_curve(scores, best_k, run_cfg)
    else:
        print(f"Scores DataFrame missing 'silhouette' or 'k' columns.")
        print(f"Available columns: {list(scores.columns)}")

K-sweep scores:


,k,inertia,silhouette,calinski,davies_bouldin
0,2,1307.786151,0.200825,49.005399,2.194470
1,3,1130.648223,0.194964,51.826414,1.768788
2,4,1009.032756,0.200336,50.639312,1.546793
3,5,901.528410,0.189053,51.280415,1.458481
4,6,803.862713,0.190386,53.095860,1.351913
5,7,733.346610,0.187172,53.098143,1.340669
6,8,683.360375,0.185104,51.770494,1.293089
7,9,639.812469,0.184949,50.728794,1.282215
8,10,604.187343,0.179702,49.515348,1.275854
9,11,568.368405,0.184217,49.057628,1.264205



Best k (max silhouette): 2


2026-03-10 14:29:32 | INFO     | market_regime.plotting | Saved plot: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/plots/03_elbow_curves.png


## PCA Scatter Plot

Two scatter panels: PC1 vs PC2, and PC3 vs PC4.
Points are coloured by `balanced_cluster` assignment.
Well-separated clusters in PC space indicate a clean macro regime structure.

In [6]:
if pca_df is not None and labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_pca_scatter(pca_df, labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_pca_scatter(pca_df, labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available columns: {list(labels.columns)}")

2026-03-10 14:29:33 | INFO     | market_regime.plotting | Saved plot: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/plots/03_pca_scatter.png


## Cluster Sizes

Bar chart showing how many quarters fall into each cluster.
The balanced clustering constrains cluster sizes to be within ±2 of the bucket size.

In [7]:
if labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_cluster_sizes(labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_cluster_sizes(labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available columns: {list(labels.columns)}")

2026-03-10 14:29:33 | INFO     | market_regime.plotting | Saved plot: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/plots/03_cluster_sizes.png


## Cluster Value Counts

Exact quarter counts for both cluster assignment strategies.

In [8]:
if labels is not None:
    if "cluster" in labels.columns:
        print("=== cluster (silhouette-selected k) ===")
        print(labels["cluster"].value_counts().sort_index().to_string())
        print()

    if "balanced_cluster" in labels.columns:
        print("=== balanced_cluster (size-constrained k) ===")
        print(labels["balanced_cluster"].value_counts().sort_index().to_string())
        print()

    print("=== Sample rows ===")
    display(labels.head(10))

=== cluster (silhouette-selected k) ===
cluster
0    107
1    197

=== balanced_cluster (size-constrained k) ===
balanced_cluster
0    58
1    62
2    62
3    60
4    62

=== Sample rows ===


,cluster,balanced_cluster,market_code
date,,,
1950-03-31,0,2,1.0
1950-06-30,0,2,1.0
1950-09-30,0,2,1.0
1950-12-31,0,0,1.0
1951-03-31,0,0,1.0
1951-06-30,1,0,1.0
1951-09-30,1,0,1.0
1951-12-31,1,4,0.0
1952-03-31,1,1,0.0
